# 11 — Failure Points & Debugging: 10 Ways Fine-Tuning Goes Wrong

> Part of the [SLM Fine-Tuning Tutorial](https://github.com/Kumarjit-Pathak/finetuning-slm)

Every fine-tuning project hits at least one of these. Learn to recognize, diagnose, and fix them before they silently sink your results.

| # | Failure Mode | Severity | Technique Most Affected |
|---|---|---|---|
| 1 | **Catastrophic forgetting** | 🔴 Critical | Full FT, sequential fine-tuning |
| 2 | **Overfitting on small data** | 🔴 Critical | All techniques |
| 3 | **Learning rate disasters** | 🟠 High | All techniques |
| 4 | **Tokenizer mismatch / padding bugs** | 🔴 Critical | All techniques |
| 5 | **Data contamination** | 🟠 High | Evaluation phase |
| 6 | **Evaluation metric traps** | 🟡 Medium | Summarization, generation |
| 7 | **MPS / Apple Silicon gotchas** | 🟠 High | All (on Mac) |
| 8 | **Quantization accuracy loss** | 🟡 Medium | QLoRA |
| 9 | **Chat template errors** | 🔴 Critical | Instruction tuning |
| 10 | **Gradient accumulation math** | 🟡 Medium | Large effective batch training |

In [1]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q",
                    "torch", "transformers", "peft", "trl",
                    "datasets", "accelerate", "rouge-score", "matplotlib"], check=True)
    print("Colab packages installed.")
else:
    print("Local environment — skipping pip install.")

Local environment — skipping pip install.


## Diagnostic Framework

The pattern for every failure mode in this notebook:

```
SYMPTOM → ROOT CAUSE → DEMONSTRATE (reproduce it) → FIX → VERIFY (it's gone)
```

### Why these failures are insidious

Most fine-tuning failures are **silent** — the code runs without errors, training loss decreases, and you only discover the problem at deployment or during evaluation.

| Failure Category | What Makes It Hard to Catch |
|---|---|
| **Training dynamics** (FP1–FP3) | Metrics look fine during training |
| **Data & format** (FP4, FP5, FP9) | Code runs; model outputs something plausible |
| **Evaluation** (FP5, FP6) | Numbers look good; problem is in interpretation |
| **Platform** (FP7, FP10) | Subtle numerical differences or silent inefficiencies |

### Shared Experiment Setup

All demos use `SmolLM2-135M-Instruct` with a small Q&A + summarization dataset. The model is simple enough that failures are obvious and quick to demonstrate.

In [2]:
import os, gc, copy, time, warnings
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=UserWarning)

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"]        = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.manual_seed(42)

# ── Device ──
device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available()          else "cpu"
)
print(f"Device: {device}")

# ── Model ──
CACHE = os.path.expanduser("~/.cache/huggingface/hub")
MODEL = None
for c in ["HuggingFaceTB/SmolLM2-360M-Instruct", "HuggingFaceTB/SmolLM2-135M-Instruct"]:
    if os.path.isdir(os.path.join(CACHE, "models--" + c.replace("/", "--"))):
        MODEL = c; break
assert MODEL, "No cached SmolLM2 found — run notebook 00 first."
print(f"Model : {MODEL}")

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW

# ── Shared utilities ──
def load_base():
    return AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device)

def free_memory(*args):
    for o in args: del o
    gc.collect()
    if device.type == "mps": torch.mps.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

LORA_CFG = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
                      target_modules=["q_proj", "v_proj"],
                      task_type=TaskType.CAUSAL_LM)
MAX_LEN = 128

# ── Mini-datasets (8 train, 4 val per task) ──
QA_TRAIN = [
    {"input": "Q: What is gradient descent?\nA:", "target": " Optimizer: w ← w − η·∇L, moves weights in steepest-descent direction."},
    {"input": "Q: What is a transformer?\nA:", "target": " Architecture: Attention(Q,K,V)=softmax(QK^T/√d_k)·V, processes sequences in parallel."},
    {"input": "Q: What is LoRA?\nA:", "target": " Low-rank adaptation: ΔW=B·A, rank r≪d, reduces trainable params 100×."},
    {"input": "Q: What is fine-tuning?\nA:", "target": " Continues pre-training on task data with low η to adapt representations."},
    {"input": "Q: What is overfitting?\nA:", "target": " train_loss≪val_loss; model memorized examples, fails to generalize."},
    {"input": "Q: What is dropout?\nA:", "target": " Zeros p fraction of activations; survivors rescaled 1/(1−p) during training."},
    {"input": "Q: What is quantization?\nA:", "target": " Maps float32→int8: q=round(w/scale), cuts model size ~4×."},
    {"input": "Q: What is self-attention?\nA:", "target": " Each token attends all others via Q·K^T/√d_k softmax; weighted V sum."},
]
QA_VAL = [
    {"input": "Q: What is batch normalization?\nA:", "target": " Normalizes z=(x−μ_B)/σ_B per batch, then scales with learned γ,β."},
    {"input": "Q: What is a residual connection?\nA:", "target": " Skip: y=F(x)+x, allows ∂L/∂x=(1+∂F/∂x)·∂L/∂y, prevents vanishing gradients."},
    {"input": "Q: What is weight decay?\nA:", "target": " L2 penalty: loss+=λ·‖w‖², shrinks weights toward 0, reduces overfitting."},
    {"input": "Q: What is perplexity?\nA:", "target": " PPL=exp(−(1/N)·Σ log p(t_i)), model's average branching factor."},
]

SUM_TRAIN = [
    {"input": "Summarize: Transformers use self-attention to process all tokens in parallel, enabling efficient training and scaling to billions of parameters.\n\nSummary:", "target": " Transformers process sequences in parallel via self-attention, enabling large-scale training."},
    {"input": "Summarize: LoRA learns low-rank weight updates B·A instead of modifying full weight matrices, reducing trainable parameters dramatically.\n\nSummary:", "target": " LoRA learns low-rank updates, cutting trainable parameters with minimal quality loss."},
    {"input": "Summarize: Quantization reduces model size by storing weights in lower precision formats like int8, enabling deployment on devices with limited memory.\n\nSummary:", "target": " Quantization compresses models to lower precision, enabling deployment on memory-constrained devices."},
    {"input": "Summarize: DPO trains models to prefer chosen over rejected responses using a closed-form loss, eliminating the need for a separate reward model.\n\nSummary:", "target": " DPO optimizes preferences directly without a reward model."},
    {"input": "Summarize: Knowledge distillation transfers capabilities from a large teacher model to a smaller student by training on the teacher's soft output distributions.\n\nSummary:", "target": " Distillation transfers teacher knowledge to a smaller student via soft labels."},
    {"input": "Summarize: RLHF aligns language models through supervised fine-tuning, reward modeling on human preferences, and PPO-based optimization.\n\nSummary:", "target": " RLHF aligns models via SFT, preference-based reward modeling, and PPO."},
    {"input": "Summarize: Model merging combines fine-tuned adapters by interpolating their weight deltas, creating a multi-task model without additional training.\n\nSummary:", "target": " Model merging interpolates weight deltas to create a multi-task model without retraining."},
    {"input": "Summarize: Prefix tuning trains only soft prefix tokens prepended to a frozen model, achieving strong performance with 0.1% of trainable parameters.\n\nSummary:", "target": " Prefix tuning trains only soft prefix tokens, adapting frozen models with minimal parameters."},
]
SUM_VAL = SUM_TRAIN[-2:]  # reuse last 2 as val for quick checks

# ── Shared training & eval functions ──
def compute_val_loss(model, dataset):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for item in dataset:
            enc = tokenizer(item["input"] + item["target"], return_tensors="pt",
                            max_length=MAX_LEN, truncation=True).to(device)
            p_len = tokenizer(item["input"], return_tensors="pt",
                              max_length=MAX_LEN, truncation=True).input_ids.shape[1]
            labels = enc.input_ids.clone(); labels[0, :p_len] = -100
            if (labels != -100).sum() == 0: continue
            total += model(**enc, labels=labels).loss.item(); n += 1
    return total / n if n else float("inf")

def quick_train(model, dataset, epochs=3, lr=1e-4, clip=1.0):
    """Minimal SGD training loop. Returns list of per-epoch losses."""
    model.train()
    opt = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    losses = []
    for _ in range(epochs):
        ep = 0.0
        for item in dataset:
            enc = tokenizer(item["input"] + item["target"], return_tensors="pt",
                            max_length=MAX_LEN, truncation=True).to(device)
            p_len = tokenizer(item["input"], return_tensors="pt",
                              max_length=MAX_LEN, truncation=True).input_ids.shape[1]
            labels = enc.input_ids.clone(); labels[0, :p_len] = -100
            loss = model(**enc, labels=labels).loss
            loss.backward()
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step(); opt.zero_grad()
            ep += loss.item()
        losses.append(ep / len(dataset))
    model.eval()
    return losses

print(f"QA  : {len(QA_TRAIN)} train, {len(QA_VAL)} val")
print(f"Sum : {len(SUM_TRAIN)} train, {len(SUM_VAL)} val")
print("Setup complete ✓")

Device: mps
Model : HuggingFaceTB/SmolLM2-135M-Instruct
QA  : 8 train, 4 val
Sum : 8 train, 2 val
Setup complete ✓


## Failures 1–3: Training Dynamics

### FP1 — Catastrophic Forgetting
**Symptom**: After sequentially fine-tuning on a new task, performance on the original task collapses.  
**Root cause**: The same adapter weights encode both tasks. High-LR Task B training overwrites the Task A gradient direction — new task's loss landscape dominates.

```
Before:  W_adapter = optimal for Task A
After (high LR):  W_adapter ≈ optimal for Task B  ← Task A forgotten
After (low LR):   W_adapter ≈ Task A + small Task B correction ✅
```

**Fix**: Use a much lower learning rate for the second fine-tuning phase (10–100× lower than original).

---

### FP2 — Overfitting on Small Data
**Symptom**: Training loss → 0, validation loss keeps rising. Model generates training examples verbatim.  
**Root cause**: Model capacity >> dataset size. LoRA with r=8 has ~500K trainable params; 4 training examples have ~500 tokens total. The adapter memorizes tokens, not patterns.

**Fix**: Reduce epochs, increase dropout, add weight decay, collect more data.

---

### FP3 — Learning Rate Disasters
**Symptom**: LR too low → no learning. LR too high → NaN loss or wild oscillations.  
**Root cause**: Gradient updates are ΔW = -η·∇L. If η is too large, the step overshoots the local minimum and loss explodes.

```
η = 1e-6: Δ per step ≈ 0.000001 × gradient → 10,000 steps to converge
η = 1e-4: Δ per step ≈ appropriate → converges in ~100 steps
η = 0.1:  Δ per step ≈ 100× gradient → overshoots → NaN
```

**Fix**: Use LR finder pattern (warm up from 1e-8, find steepest loss descent). Rule of thumb: start at `1e-4` for LoRA, `5e-5` for full FT.

In [3]:
# ══════════════════════════════════════════════════════════════
#  FP1: CATASTROPHIC FORGETTING
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("FP1: CATASTROPHIC FORGETTING")
print("=" * 60)

# Step 1: Train LoRA adapter on Task A (QA)
base_fp1 = load_base()
lora_A = get_peft_model(base_fp1, LORA_CFG)
quick_train(lora_A, QA_TRAIN, epochs=3, lr=1e-4)
qa_before = compute_val_loss(lora_A, QA_VAL)
print(f"  After QA training (Task A):            QA val loss = {qa_before:.4f}")

# Step 2: Continue training on Task B (SUM) with HIGH learning rate → forget Task A
forget_model = copy.deepcopy(lora_A)
quick_train(forget_model, SUM_TRAIN, epochs=5, lr=5e-3, clip=0.0)  # high LR, no clipping
qa_after_forget = compute_val_loss(forget_model, QA_VAL)
print(f"  After SUM fine-tune (HIGH LR=5e-3): QA val loss = {qa_after_forget:.4f}  🔴 +{qa_after_forget-qa_before:.3f}")
free_memory(forget_model)

# Step 3: Fix — low LR for second fine-tuning phase
fixed_model = copy.deepcopy(lora_A)
quick_train(fixed_model, SUM_TRAIN, epochs=3, lr=5e-5)
qa_after_fixed = compute_val_loss(fixed_model, QA_VAL)
print(f"  After SUM fine-tune (LOW  LR=5e-5):  QA val loss = {qa_after_fixed:.4f}  ✅ Δ={qa_after_fixed-qa_before:+.3f}")
free_memory(base_fp1, lora_A, fixed_model)

fp1_results = (qa_before, qa_after_forget, qa_after_fixed)

# ══════════════════════════════════════════════════════════════
#  FP2: OVERFITTING ON SMALL DATA
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FP2: OVERFITTING ON SMALL DATA")
print("=" * 60)

TINY_TRAIN = QA_TRAIN[:4]   # only 4 training examples
base_fp2   = load_base()
lora_fp2   = get_peft_model(base_fp2, LORA_CFG)
lora_fp2.train()
opt_fp2    = AdamW(lora_fp2.parameters(), lr=2e-4, weight_decay=0.0)  # no weight decay on purpose

train_losses_fp2, val_losses_fp2 = [], []
print(f"  {'Epoch':>5}  {'Train loss':>11}  {'Val loss':>10}")
for ep in range(25):
    ep_loss = 0.0
    for item in TINY_TRAIN:
        enc = tokenizer(item["input"] + item["target"], return_tensors="pt",
                        max_length=MAX_LEN, truncation=True).to(device)
        p_len = tokenizer(item["input"], return_tensors="pt",
                          max_length=MAX_LEN, truncation=True).input_ids.shape[1]
        labels = enc.input_ids.clone(); labels[0, :p_len] = -100
        loss = lora_fp2(**enc, labels=labels).loss
        loss.backward()
        opt_fp2.step(); opt_fp2.zero_grad()
        ep_loss += loss.item()
    train_loss = ep_loss / len(TINY_TRAIN)
    val_loss   = compute_val_loss(lora_fp2, QA_VAL)
    train_losses_fp2.append(train_loss)
    val_losses_fp2.append(val_loss)
    if (ep + 1) % 5 == 0:
        flag = "🔴" if val_loss > val_losses_fp2[0] + 0.3 else ""
        print(f"  {ep+1:>5}  {train_loss:>11.4f}  {val_loss:>10.4f} {flag}")

free_memory(base_fp2, lora_fp2, opt_fp2)

# ══════════════════════════════════════════════════════════════
#  FP3: LEARNING RATE DISASTERS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FP3: LEARNING RATE DISASTERS")
print("=" * 60)

lr_values  = [1e-6, 1e-5, 1e-4, 5e-4, 1e-2, 0.5]
lr_results = []
base_lr_loss = None

for lr in lr_values:
    base_lr = load_base()
    lora_lr = get_peft_model(base_lr, LORA_CFG)
    if base_lr_loss is None:
        base_lr_loss = compute_val_loss(lora_lr, QA_VAL)
    lora_lr.train()
    opt_lr = AdamW(lora_lr.parameters(), lr=lr)
    final_loss = base_lr_loss
    crashed = False
    for item in QA_TRAIN[:6]:   # 6 steps only
        enc = tokenizer(item["input"] + item["target"], return_tensors="pt",
                        max_length=MAX_LEN, truncation=True).to(device)
        p_len = tokenizer(item["input"], return_tensors="pt",
                          max_length=MAX_LEN, truncation=True).input_ids.shape[1]
        labels = enc.input_ids.clone(); labels[0, :p_len] = -100
        loss = lora_lr(**enc, labels=labels).loss
        if torch.isnan(loss) or torch.isinf(loss):
            crashed = True; break
        loss.backward()
        opt_lr.step(); opt_lr.zero_grad()
    if not crashed:
        final_loss = compute_val_loss(lora_lr, QA_VAL)
    lr_results.append((lr, final_loss, crashed))
    free_memory(base_lr, lora_lr, opt_lr)

print(f"  {'LR':<12} {'Val loss after 6 steps':>22}  Note")
print(f"  {'-'*55}")
for lr, loss, crashed in lr_results:
    note = "🔴 DIVERGED" if crashed else ("🔴 no learning" if loss >= base_lr_loss - 0.02 else "✅")
    print(f"  {lr:<12.0e} {loss:>22.4f}  {note}")

FP1: CATASTROPHIC FORGETTING


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  After QA training (Task A):            QA val loss = 4.6051
  After SUM fine-tune (HIGH LR=5e-3): QA val loss = 5.9959  🔴 +1.391
  After SUM fine-tune (LOW  LR=5e-5):  QA val loss = 4.6031  ✅ Δ=-0.002

FP2: OVERFITTING ON SMALL DATA


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  Epoch   Train loss    Val loss
      5       4.8002      4.4862 
     10       4.0901      4.2276 
     15       3.2344      4.0536 
     20       2.2285      4.1505 
     25       1.1052      4.8754 

FP3: LEARNING RATE DISASTERS


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  LR           Val loss after 6 steps  Note
  -------------------------------------------------------
  1e-06                        4.7656  🔴 no learning
  1e-05                        4.7627  🔴 no learning
  1e-04                        4.7333  ✅
  5e-04                        4.5930  ✅
  1e-02                        4.6455  ✅
  5e-01                       12.6628  🔴 no learning


## Failures 4, 5, 9: Data & Format

### FP4 — Tokenizer Mismatch / Padding Bugs
**Symptom**: Batched training crashes with shape mismatches; or silently produces wrong results because padding tokens are treated as real tokens.  
**Root cause**: Two sub-problems:
1. **Pad token not set**: `transformers` uses `tokenizer.pad_token` for left-padding. If it's `None`, batching fails.
2. **Labels include padding**: If `labels` include `pad_token_id` positions, the model penalizes predicting padding → corrupted gradients.

**Fix**: Always set `pad_token = eos_token`; always mask padding positions in labels with `-100`.

---

### FP5 — Data Contamination
**Symptom**: Evaluation metrics look suspiciously good. Model "knows" the answers verbatim.  
**Root cause**: Test/val examples accidentally appear in the training set. Classic example: using `train_test_split` on chronologically ordered data, or forgetting to de-duplicate before splitting.

**Fix**: Always split BEFORE tokenization; use hash-based deduplication; keep a held-out test set never used during development.

---

### FP9 — Chat Template Errors
**Symptom**: Instruction-tuned model produces incoherent outputs, ignores instructions, or repeats the prompt.  
**Root cause**: The model was RLHF-trained with a specific chat template (e.g., `<|im_start|>user\n...<|im_end|>\n`). When you fine-tune or evaluate without this template, the model sees a completely different input distribution.

```python
# WRONG (raw format)
"What is LoRA?"  → model confused, generates random text

# RIGHT (chat template)
tokenizer.apply_chat_template([{"role":"user","content":"What is LoRA?"}],
                               tokenize=False, add_generation_prompt=True)
# "<|im_start|>user\nWhat is LoRA?<|im_end|>\n<|im_start|>assistant\n"
```

**Fix**: Always use `apply_chat_template` for instruction-tuned models. For raw base models (not instruction-tuned), skip the template.

In [7]:
# ══════════════════════════════════════════════════════════════
#  FP4: TOKENIZER MISMATCH / PADDING BUGS
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("FP4: TOKENIZER MISMATCH / PADDING BUGS")
print("=" * 60)

# Scenario A: pad_token not set → batching fails
tok_bad = AutoTokenizer.from_pretrained(MODEL)
tok_bad.pad_token = None   # simulate forgotten setup

texts = ["Short text.", "A much longer piece of text that will need padding."]
try:
    enc_bad = tok_bad(texts, return_tensors="pt", padding=True)
    print("  No error (would silently pad with 0 = unk token)")
except Exception as e:
    print(f"  🔴 Error without pad_token: {type(e).__name__}: {e}")

# Fix: set pad_token = eos_token
tok_good = AutoTokenizer.from_pretrained(MODEL)
tok_good.pad_token = tok_good.eos_token
enc_good = tok_good(texts, return_tensors="pt", padding=True)
print(f"  ✅ With pad_token set: input_ids shape = {enc_good.input_ids.shape}")

# Scenario B: labels include pad positions → corrupted gradients
print("\n  Padding position demo (why we mask labels with -100):")
sample = texts[0]
enc_padded = tok_good([sample, sample + " " * 30], return_tensors="pt", padding=True)
labels_wrong = enc_padded.input_ids.clone()
labels_right  = enc_padded.input_ids.clone()
# Mask padding tokens in correct version
labels_right[enc_padded.attention_mask == 0] = -100
n_wrong_loss_tokens = (labels_wrong != -100).sum().item()
n_right_loss_tokens = (labels_right  != -100).sum().item()
print(f"  Labels WITHOUT masking: {n_wrong_loss_tokens} tokens contribute to loss (includes padding)")
print(f"  Labels WITH    masking: {n_right_loss_tokens} tokens contribute to loss (only real tokens) ✅")

# ══════════════════════════════════════════════════════════════
#  FP5: DATA CONTAMINATION
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FP5: DATA CONTAMINATION")
print("=" * 60)

# Train a model on QA_TRAIN with MANY epochs to memorize training data
base_fp5 = load_base()
lora_fp5 = get_peft_model(base_fp5, LORA_CFG)
quick_train(lora_fp5, QA_TRAIN, epochs=15, lr=2e-4)  # many epochs → memorization

loss_clean        = compute_val_loss(lora_fp5, QA_VAL)         # proper holdout
loss_contaminated = compute_val_loss(lora_fp5, QA_TRAIN[:4])  # training examples as "eval"

gap = loss_clean - loss_contaminated
print(f"  Val loss on clean holdout (4 unseen examples): {loss_clean:.4f}")
print(f"  Val loss on CONTAMINATED set (training data):  {loss_contaminated:.4f}  {'🔴 suspiciously low!' if gap > 0.05 else '(small gap — model not fully memorized)'}")
print(f"  Optimism bias: {gap:.4f} val loss units {'← model looks better than it really is!' if gap > 0.05 else ''}")
free_memory(base_fp5, lora_fp5)

# ══════════════════════════════════════════════════════════════
#  FP9: CHAT TEMPLATE ERRORS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FP9: CHAT TEMPLATE ERRORS")
print("=" * 60)

question = "What is LoRA?"

# Raw format (no chat template)
raw_prompt = f"Q: {question}\nA:"
raw_tokens  = tokenizer(raw_prompt, return_tensors="pt").to(device)

# Chat template format
chat_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": question}],
    tokenize=False, add_generation_prompt=True
)
chat_tokens = tokenizer(chat_prompt, return_tensors="pt").to(device)

print(f"  RAW  format ({raw_tokens.input_ids.shape[1]:2d} tokens): {repr(raw_prompt)}")
print(f"  CHAT format ({chat_tokens.input_ids.shape[1]:2d} tokens): {repr(chat_prompt[:80])}...")

# Show output difference
base_demo = load_base()
base_demo.eval()
with torch.no_grad():
    raw_out  = base_demo.generate(**raw_tokens,  max_new_tokens=15, do_sample=False)
    chat_out = base_demo.generate(**chat_tokens, max_new_tokens=15, do_sample=False)

raw_decoded  = tokenizer.decode(raw_out[0][raw_tokens.input_ids.shape[1]:],  skip_special_tokens=True)
chat_decoded = tokenizer.decode(chat_out[0][chat_tokens.input_ids.shape[1]:], skip_special_tokens=True)
print(f"\n  RAW  response  : {repr(raw_decoded[:60])}")
print(f"  CHAT response  : {repr(chat_decoded[:60])}")
print(f"\n  ✅ For instruction-tuned models, always use apply_chat_template()")
free_memory(base_demo)

FP4: TOKENIZER MISMATCH / PADDING BUGS
  🔴 Error without pad_token: ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.
  ✅ With pad_token set: input_ids shape = torch.Size([2, 11])

  Padding position demo (why we mask labels with -100):
  Labels WITHOUT masking: 10 tokens contribute to loss (includes padding)
  Labels WITH    masking: 8 tokens contribute to loss (only real tokens) ✅

FP5: DATA CONTAMINATION


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  Val loss on clean holdout (4 unseen examples): 4.2730
  Val loss on CONTAMINATED set (training data):  2.2094  🔴 suspiciously low!
  Optimism bias: 2.0636 val loss units ← model looks better than it really is!

FP9: CHAT TEMPLATE ERRORS
  RAW  format (10 tokens): 'Q: What is LoRA?\nA:'
  CHAT format (35 tokens): '<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Huggi'...


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


  RAW  response  : ' LoRa is a low-power radio frequency (RF) technology that us'
  CHAT response  : 'LoRa is a low-power radio frequency (LPF) technology that'

  ✅ For instruction-tuned models, always use apply_chat_template()


## Failures 6 & 8: Evaluation Traps & Quantization

### FP6 — Evaluation Metric Traps
**Symptom**: ROUGE/BLEU scores look excellent, but the generated text is clearly wrong or useless.  
**Root cause**: n-gram overlap metrics don't measure factual correctness, relevance, or coherence. They reward surface similarity, which can be gamed by repetition or verbosity.

| Summary type | ROUGE-1 | Quality |
|---|---|---|
| Verbatim copy of input | Very high (0.8+) | Terrible — too long, not condensed |
| Empty string | 0.0 | Terrible — no content |
| Concise accurate summary | Moderate (0.3–0.5) | ✅ Correct and useful |

**Fix**: Always use 2–3 complementary metrics (ROUGE + BERTScore + human inspection). For generation quality, treat ROUGE as a lower bound signal, not a quality guarantee.

---

### FP8 — Quantization Accuracy Loss
**Symptom**: After quantization (int8/int4), perplexity jumps and specific outputs change.  
**Root cause**: Quantization introduces rounding error: `round(w / scale) * scale ≠ w`. Errors compound across layers through matrix multiplications.

```
float32 matmul: exact floating point arithmetic
int8 matmul: quantize → multiply integers → dequantize → error ≈ |Δ| ≈ scale/2
```

For weights with range [-1, 1] in float32:
- int8 scale = 1/127 ≈ 0.0079  → max rounding error ≈ 0.004 per weight
- int4 scale = 1/7   ≈ 0.143   → max rounding error ≈ 0.07 per weight (18× worse)

**Fix**: Use NF4 (Normal Float 4) instead of uniform int4; apply absmax quantization per group (group_size=128); use double quantization for quantizing the scale factors themselves.

In [ ]:
from rouge_score import rouge_scorer

# ══════════════════════════════════════════════════════════════
#  FP6: EVALUATION METRIC TRAPS (ROUGE)
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("FP6: EVALUATION METRIC TRAP — ROUGE SCORES")
print("=" * 60)

reference = (
    "Transformers use parallel self-attention to process sequences, "
    "enabling efficient large-scale training and strong performance."
)
source_doc = (
    "Transformers are a type of neural network architecture that process "
    "entire sequences in parallel using self-attention mechanisms, where "
    "each token attends to every other token simultaneously. This parallelism "
    "makes training on modern hardware extremely efficient and has enabled "
    "scaling to billions of parameters, resulting in strong performance "
    "across many natural language processing tasks."
)

candidates = {
    "A — Verbatim copy of input doc": source_doc,
    "B — Empty output":               "",
    "C — Concise accurate summary":   "Transformers use parallel self-attention for efficient training and scale.",
    "D — Wrong but similar n-grams":  "Transformers use parallel attention mechanisms that process sequences in training.",
    "E — Correct but different words": "Self-attention allows transformers to process all tokens at once, making training fast.",
}

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
print(f"\n  {'Candidate':<40} {'R-1':>6} {'R-2':>6} {'R-L':>6}  Quality")
print(f"  {'-'*70}")
for name, cand in candidates.items():
    if cand == "":
        scores = {"rouge1": type("", (), {"fmeasure": 0.0})(),
                  "rouge2": type("", (), {"fmeasure": 0.0})(),
                  "rougeL": type("", (), {"fmeasure": 0.0})()}
    else:
        scores = scorer.score(reference, cand)
    r1 = scores["rouge1"].fmeasure
    r2 = scores["rouge2"].fmeasure
    rl = scores["rougeL"].fmeasure
    quality = "✅ Good" if name.startswith("C") or name.startswith("E") else "🔴 Bad"
    print(f"  {name:<40} {r1:>6.3f} {r2:>6.3f} {rl:>6.3f}  {quality}")
print(f"\n  ⚠️  Candidate D (wrong facts, similar n-grams) scores higher than Candidate E (correct different words)!")
print(f"  ⚠️  ROUGE rewards n-gram overlap, NOT factual correctness.")
print(f"  ⚠️  ROUGE-Recall (not shown) gives verbatim copies a score of ~1.0 — always use F-measure.")
print(f"  ✅  Always combine ROUGE-F + BERTScore + factual consistency + human inspection.")

# ══════════════════════════════════════════════════════════════
#  FP8: QUANTIZATION ACCURACY LOSS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FP8: QUANTIZATION ACCURACY LOSS")
print("=" * 60)

def simulate_quantization(tensor, bits=8):
    """Simulate absmax uniform quantization."""
    absmax = tensor.abs().max()
    scale  = absmax / (2 ** (bits - 1) - 1)
    q      = torch.round(tensor / scale).clamp(-(2**(bits-1)), 2**(bits-1)-1)
    return q * scale   # dequantized

torch.manual_seed(0)
W = torch.randn(64, 64)           # simulate a weight matrix
x = torch.randn(64)               # input vector

# float32 result (ground truth)
out_f32 = W @ x

# int8 quantized
W_int8   = simulate_quantization(W, bits=8)
out_int8 = W_int8 @ x

# int4 quantized (simulated)
W_int4   = simulate_quantization(W, bits=4)
out_int4 = W_int4 @ x

err_int8 = (out_f32 - out_int8).abs().mean().item()
err_int4 = (out_f32 - out_int4).abs().mean().item()
rel_int8 = err_int8 / out_f32.abs().mean().item() * 100
rel_int4 = err_int4 / out_f32.abs().mean().item() * 100

print(f"  Single-layer matmul error (64×64 weight matrix):")
print(f"    float32 (baseline):  max diff = 0.000000  (0.00%)")
print(f"    int8    quantized:   mean abs err = {err_int8:.6f}  ({rel_int8:.2f}% relative)")
print(f"    int4    quantized:   mean abs err = {err_int4:.6f}  ({rel_int4:.2f}% relative)  🔴 ~{rel_int4/rel_int8:.0f}× worse than int8")

# Cumulate over N layers
N_LAYERS = 30  # SmolLM2-135M has 30 transformer layers
print(f"\n  Cumulative effect over {N_LAYERS} transformer layers (approximation):")
print(f"    int8: SNR degrades per layer; ~{N_LAYERS}× amplification of single-layer error")
print(f"    int4: More aggressive compression → noticeable PPL increase (~0.5–2.0 PPL pts)")
print(f"\n  ✅ Fix: Use NF4 quantization (designed for normally-distributed weights)")
print(f"     + double quantization + group_size=128 (QLoRA approach)")

FP6: EVALUATION METRIC TRAP — ROUGE SCORES

  Candidate                                   R-1    R-2    R-L  Quality
  ----------------------------------------------------------------------
  A — Verbatim copy of input doc            0.429  0.059  0.257  🔴 Bad
  B — Empty output                          0.000  0.000  0.000  🔴 Bad
  C — Concise accurate summary              0.692  0.417  0.615  ✅ Good
  D — Wrong but similar n-grams             0.538  0.250  0.538  🔴 Bad
  E — Correct but different words           0.414  0.148  0.345  ✅ Good

  ⚠️  Candidate A (verbatim copy) outscores correct summaries on ROUGE!
  ⚠️  Candidate D (wrong facts, similar n-grams) also scores high.
  ✅  Always combine ROUGE with factual accuracy checks and human review.

FP8: QUANTIZATION ACCURACY LOSS
  Single-layer matmul error (64×64 weight matrix):
    float32 (baseline):  max diff = 0.000000  (0.00%)
    int8    quantized:   mean abs err = 0.058669  (0.98% relative)
    int4    quantized:   mean abs e

## Failures 7 & 10: Platform & Optimization

### FP7 — MPS / Apple Silicon Gotchas
Running PyTorch on Apple Silicon MPS is mostly transparent, but several common pitfalls break silently:

| Issue | Symptom | Fix |
|---|---|---|
| `torch.float16` on MPS | `RuntimeError: MPS does not support float16` | Use `torch.float32` or `torch.bfloat16` |
| `torch.autocast("mps")` | Silent no-op on older PyTorch | Upgrade to PyTorch ≥ 2.2 or skip autocast |
| Missing `torch.mps.empty_cache()` | Memory never freed between experiments | Call after each `del model; gc.collect()` |
| Unsupported MPS ops (e.g., some einsum variants) | Silently falls back to CPU → slow | Check `torch.backends.mps.is_available()` first |
| Non-determinism on MPS | Results vary between runs even with `manual_seed` | MPS kernel scheduling is non-deterministic |

---

### FP10 — Gradient Accumulation Math
**Symptom**: Using gradient accumulation produces different effective loss than the equivalent larger batch size.  
**Root cause**: Two common mistakes:

1. **Loss is not divided by accumulation steps**: Cross-entropy loss is averaged per token inside `model(...)`. When you accumulate gradients over N steps, you're summing N averaged losses. This is mathematically equivalent to averaging, **but** if you add an explicit `loss / N` scaling, the effective gradient is 1/N of what it should be.

2. **Scheduler steps vs optimizer steps**: If your LR scheduler steps every `optimizer.step()` call, and you have `grad_accum=4`, your scheduler fires 4× too slowly relative to the training progress.

```python
# CORRECT gradient accumulation pattern:
for step, batch in enumerate(dataloader):
    loss = model(**batch).loss          # mean over tokens
    loss.backward()                     # accumulate gradients
    if (step + 1) % GRAD_ACCUM == 0:
        optimizer.step()
        scheduler.step()               # ← step scheduler HERE, not per batch
        optimizer.zero_grad()
```

In [6]:
# ══════════════════════════════════════════════════════════════
#  FP7: MPS / APPLE SILICON GOTCHAS
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("FP7: MPS / APPLE SILICON GOTCHAS")
print("=" * 60)

# Issue 1: float16 on MPS
print("\n  Issue 1 — torch.float16 on MPS:")
if device.type == "mps":
    try:
        bad_tensor = torch.tensor([1.0, 2.0], dtype=torch.float16, device=device)
        _ = bad_tensor @ bad_tensor
        print("  Ran without error (PyTorch version may support f16 on MPS now)")
    except RuntimeError as e:
        print(f"  🔴 float16 error: {str(e)[:80]}")
    f32_tensor = torch.tensor([1.0, 2.0], dtype=torch.float32, device=device)
    _ = f32_tensor @ f32_tensor
    print(f"  ✅ float32 on MPS: works fine — always use dtype=torch.float32")
else:
    print(f"  (Running on {device}, not MPS — skipping MPS-specific test)")

# Issue 2: Memory management
print("\n  Issue 2 — Memory management on MPS:")
if device.type == "mps":
    dummy = torch.randn(100, 100, device=device)
    before = torch.mps.current_allocated_memory()
    del dummy
    import gc; gc.collect()
    after_no_flush  = torch.mps.current_allocated_memory()
    torch.mps.empty_cache()
    after_flush = torch.mps.current_allocated_memory()
    print(f"  Allocated before del:       {before/1e6:.2f} MB")
    print(f"  After del (no empty_cache): {after_no_flush/1e6:.2f} MB")
    print(f"  After torch.mps.empty_cache(): {after_flush/1e6:.2f} MB ✅")
else:
    print(f"  (Skipped — running on {device})")

# Issue 3: Non-determinism
print("\n  Issue 3 — MPS Non-determinism:")
results = []
for seed in [42, 42]:
    torch.manual_seed(seed)
    x = torch.randn(10, 10, device=device)
    y = torch.randn(10, 10, device=device)
    results.append((x @ y)[0, 0].item())
same = abs(results[0] - results[1]) < 1e-6
print(f"  Same seed, same matmul: {results[0]:.6f} vs {results[1]:.6f}")
print(f"  Deterministic: {'✅ Yes' if same else '⚠️  No — MPS can be non-deterministic'}")

# ══════════════════════════════════════════════════════════════
#  FP10: GRADIENT ACCUMULATION MATH
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FP10: GRADIENT ACCUMULATION MATH")
print("=" * 60)

# Show that accumulation = large batch (when done correctly)
base_fp10 = load_base()
lora_fp10 = get_peft_model(base_fp10, LORA_CFG)

GRAD_ACCUM = 4
# Items for the "virtual" large batch (4 items)
batch_items = QA_TRAIN[:4]

def get_grads(model):
    """Return dict of param_name → gradient tensor (detached copy)."""
    return {n: p.grad.detach().clone()
            for n, p in model.named_parameters() if p.grad is not None}

# Method A: Accumulate gradients manually over 4 steps
opt_acc = AdamW(lora_fp10.parameters(), lr=1e-4)
opt_acc.zero_grad()
for i, item in enumerate(batch_items):
    enc = tokenizer(item["input"] + item["target"], return_tensors="pt",
                    max_length=MAX_LEN, truncation=True).to(device)
    p_len = tokenizer(item["input"], return_tensors="pt",
                      max_length=MAX_LEN, truncation=True).input_ids.shape[1]
    labels = enc.input_ids.clone(); labels[0, :p_len] = -100
    loss = lora_fp10(**enc, labels=labels).loss / GRAD_ACCUM  # scale loss
    loss.backward()
grads_accumulated = get_grads(lora_fp10)

# Compare accumulated gradient magnitudes
print(f"\n  Gradient accumulation ({GRAD_ACCUM} steps of batch_size=1):")
total_grad_norm = sum(g.norm().item()**2 for g in grads_accumulated.values()) ** 0.5
print(f"    Total gradient norm (accumulated): {total_grad_norm:.6f}")

# Method B: batch_size=4 in a single step (collect and average manually)
opt_acc.zero_grad()
accum_loss = 0.0
for item in batch_items:
    enc = tokenizer(item["input"] + item["target"], return_tensors="pt",
                    max_length=MAX_LEN, truncation=True).to(device)
    p_len = tokenizer(item["input"], return_tensors="pt",
                      max_length=MAX_LEN, truncation=True).input_ids.shape[1]
    labels = enc.input_ids.clone(); labels[0, :p_len] = -100
    accum_loss += lora_fp10(**enc, labels=labels).loss / GRAD_ACCUM
accum_loss.backward()
grads_batch = get_grads(lora_fp10)
total_batch_norm = sum(g.norm().item()**2 for g in grads_batch.values()) ** 0.5
print(f"    Total gradient norm (averaged batch): {total_batch_norm:.6f}")

# Check equivalence
common_keys = set(grads_accumulated) & set(grads_batch)
max_diff = max((grads_accumulated[k] - grads_batch[k]).abs().max().item()
               for k in common_keys)
print(f"    Max gradient diff between methods: {max_diff:.2e}")
print(f"    {'✅ Equivalent (max diff < 1e-5)' if max_diff < 1e-5 else '⚠️  Not equivalent!'}")

# Common mistake: forgetting loss / GRAD_ACCUM
print(f"\n  ⚠️  Common mistake: accumulating WITHOUT dividing loss by GRAD_ACCUM")
print(f"     → gradients are GRAD_ACCUM× too large → effectively LR × GRAD_ACCUM")
print(f"     → fix: always scale loss = model_loss / GRAD_ACCUM before backward()")
print(f"\n  ⚠️  Scheduler mistake: step scheduler every batch instead of every optimizer step")
print(f"     → effective LR schedule runs GRAD_ACCUM× faster than intended")
print(f"     → fix: scheduler.step() inside the `if step % GRAD_ACCUM == 0` block")
free_memory(base_fp10, lora_fp10, opt_acc)

FP7: MPS / APPLE SILICON GOTCHAS

  Issue 1 — torch.float16 on MPS:
  Ran without error (PyTorch version may support f16 on MPS now)
  ✅ float32 on MPS: works fine — always use dtype=torch.float32

  Issue 2 — Memory management on MPS:
  Allocated before del:       3864.59 MB
  After del (no empty_cache): 3864.55 MB
  After torch.mps.empty_cache(): 3864.55 MB ✅

  Issue 3 — MPS Non-determinism:
  Same seed, same matmul: -0.697523 vs -0.697523
  Deterministic: ✅ Yes

FP10: GRADIENT ACCUMULATION MATH


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


  Gradient accumulation (4 steps of batch_size=1):
    Total gradient norm (accumulated): 0.891341
    Total gradient norm (averaged batch): 0.885701
    Max gradient diff between methods: 8.08e-03
    ⚠️  Not equivalent!

  ⚠️  Common mistake: accumulating WITHOUT dividing loss by GRAD_ACCUM
     → gradients are GRAD_ACCUM× too large → effectively LR × GRAD_ACCUM
     → fix: always scale loss = model_loss / GRAD_ACCUM before backward()

  ⚠️  Scheduler mistake: step scheduler every batch instead of every optimizer step
     → effective LR schedule runs GRAD_ACCUM× faster than intended
     → fix: scheduler.step() inside the `if step % GRAD_ACCUM == 0` block


## Debugging Checklist

Use this checklist before blaming your model architecture or dataset quality.

### Pre-training
- [ ] `tokenizer.pad_token` is set (usually `= tokenizer.eos_token`)
- [ ] Labels mask prompt tokens and padding with `-100`
- [ ] `apply_chat_template` used for instruction-tuned base models
- [ ] Train/val/test split done **before** tokenization; no data leakage
- [ ] Training loss **decreasing** after first epoch (if flat → LR too low)
- [ ] Training loss **not NaN/inf** (if it is → LR too high, or mixed precision issue)

### During training
- [ ] Val loss tracked every epoch — if rising while train falls → overfitting
- [ ] LR order of magnitude: LoRA `1e-4`, Full FT `5e-5`, QLoRA `2e-4`
- [ ] On MPS: `dtype=torch.float32`; call `torch.mps.empty_cache()` between experiments
- [ ] Gradient accumulation: loss divided by `GRAD_ACCUM`; scheduler stepped per optimizer step

### Evaluation
- [ ] Evaluate on **held-out** data never seen in training or validation
- [ ] Use **2–3 complementary metrics** (not just ROUGE or just perplexity)
- [ ] Check a few outputs **manually** — metrics don't catch all failure modes
- [ ] For sequential fine-tuning: re-evaluate on **all previous tasks** after each new task

In [8]:
# ── Summary table of all 10 failure modes ──
print("=" * 80)
print("  FAILURE POINTS SUMMARY — ALL 10 MODES")
print("=" * 80)

failures = [
    ("FP1", "Catastrophic Forgetting",     "🔴 Critical", "Overwriting adapter with high-LR second task",      "Low LR for sequential training (10–100× lower)"),
    ("FP2", "Overfitting on Small Data",   "🔴 Critical", "Model capacity >> dataset size",                    "Fewer epochs, dropout, weight decay, more data"),
    ("FP3", "LR Disasters",               "🟠 High",     "η too large → diverge; too small → no learning",    "LR finder; start LoRA at 1e-4, Full FT at 5e-5"),
    ("FP4", "Tokenizer / Padding Bugs",   "🔴 Critical", "pad_token=None; padding in labels",                 "tokenizer.pad_token=eos_token; mask padding with -100"),
    ("FP5", "Data Contamination",         "🟠 High",     "Test examples in training set",                     "Split before tokenization; hash-deduplicate first"),
    ("FP6", "Eval Metric Traps",          "🟡 Medium",   "ROUGE/BLEU rewards n-gram overlap not quality",     "Use 3+ metrics; always manual-inspect outputs"),
    ("FP7", "MPS/Apple Silicon Gotchas",  "🟠 High",     "float16 errors; memory leaks; non-determinism",     "float32; mps.empty_cache(); fix seed + check ops"),
    ("FP8", "Quantization Accuracy Loss", "🟡 Medium",   "Rounding error compounds across 30 layers",         "Use NF4 + double quantization + group_size=128"),
    ("FP9", "Chat Template Errors",       "🔴 Critical", "Model trained with template, evaluated without",    "Always apply_chat_template() for instruct models"),
    ("FP10","Gradient Accumulation Math", "🟡 Medium",   "Loss not scaled; scheduler steps per batch not opt","loss/=GRAD_ACCUM; scheduler.step() per optimizer step"),
]

print(f"\n  {'#':<5} {'Failure Mode':<28} {'Sev':>12}  {'Root Cause':<35} {'Fix'}")
print(f"  {'-'*5} {'-'*28} {'-'*12}  {'-'*35} {'-'*35}")
for fp, name, sev, cause, fix in failures:
    print(f"  {fp:<5} {name:<28} {sev:>12}  {cause[:33]:<35} {fix[:50]}")

# Show FP1 numeric results
print(f"\n  FP1 Measured Results (SmolLM2-{MODEL.split('-')[1]}):")
q0, q_forget, q_fix = fp1_results
print(f"    QA val loss — before: {q0:.4f} | after forgetting: {q_forget:.4f} (+{q_forget-q0:.3f}) | fixed: {q_fix:.4f} ({q_fix-q0:+.3f})")

# Show FP5 contamination gap
print(f"  FP5 Contamination optimism bias: {loss_clean - loss_contaminated:.4f} val loss units")

# Show FP6 ROUGE trap
print(f"  FP6 ROUGE-1 trap: verbatim copy can outscore correct summary by design")

  FAILURE POINTS SUMMARY — ALL 10 MODES

  #     Failure Mode                          Sev  Root Cause                          Fix
  ----- ---------------------------- ------------  ----------------------------------- -----------------------------------
  FP1   Catastrophic Forgetting        🔴 Critical  Overwriting adapter with high-LR    Low LR for sequential training (10–100× lower)
  FP2   Overfitting on Small Data      🔴 Critical  Model capacity >> dataset size      Fewer epochs, dropout, weight decay, more data
  FP3   LR Disasters                       🟠 High  η too large → diverge; too small    LR finder; start LoRA at 1e-4, Full FT at 5e-5
  FP4   Tokenizer / Padding Bugs       🔴 Critical  pad_token=None; padding in labels   tokenizer.pad_token=eos_token; mask padding with -
  FP5   Data Contamination                 🟠 High  Test examples in training set       Split before tokenization; hash-deduplicate first
  FP6   Eval Metric Traps                🟡 Medium  ROUGE/BLEU reward

In [9]:
# ── 3-Panel Diagnostic Chart ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Failure Points & Debugging — Key Diagnostics", fontsize=14, fontweight="bold", y=1.01)

# ── Panel 1: FP1 Catastrophic Forgetting ──
ax = axes[0]
labels = ["QA only\n(baseline)", "High LR\nforgetting 🔴", "Low LR\nfix ✅"]
values = list(fp1_results)
colors = ["#4c8bf5", "#ea4335", "#34a853"]
bars = ax.bar(labels, values, color=colors, alpha=0.85, edgecolor="white", width=0.5)
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.axhline(values[0], color="#4c8bf5", linestyle="--", alpha=0.5, label=f"Baseline ({values[0]:.3f})")
ax.set_ylabel("QA Val Loss"); ax.set_title("FP1: Catastrophic Forgetting\n(lower = better)", fontweight="bold")
ax.set_ylim(min(values) * 0.95, max(values) * 1.05)
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

# ── Panel 2: FP2 Overfitting Curves ──
ax = axes[1]
epochs_range = list(range(1, len(train_losses_fp2) + 1))
ax.plot(epochs_range, train_losses_fp2, "o-", color="#4c8bf5", label="Train loss", linewidth=2, markersize=4)
ax.plot(epochs_range, val_losses_fp2,   "s-", color="#ea4335", label="Val loss",   linewidth=2, markersize=4)
# Highlight the "optimal" point (min val loss)
best_ep = np.argmin(val_losses_fp2)
ax.axvline(x=best_ep + 1, color="#fbbc04", linestyle="--", alpha=0.7, label=f"Best val (ep {best_ep+1})")
ax.fill_between(epochs_range, train_losses_fp2, val_losses_fp2,
                where=[v > t for t, v in zip(train_losses_fp2, val_losses_fp2)],
                alpha=0.1, color="#ea4335", label="Overfit gap")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("FP2: Overfitting on Small Data\n(4 examples, 25 epochs)", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── Panel 3: FP3 LR Disaster ──
ax = axes[2]
lrs_plot  = [r[0] for r in lr_results if not r[2]]
vals_plot = [r[1] for r in lr_results if not r[2]]
crash_lrs = [r[0] for r in lr_results if r[2]]

log_lrs = np.log10(lrs_plot)
bar_colors_lr = ["#aaaaaa" if v >= (base_lr_loss - 0.05) else "#34a853" for v in vals_plot]
ax.bar(range(len(lrs_plot)), vals_plot, color=bar_colors_lr, alpha=0.85, edgecolor="white")
for i, (lr, v) in enumerate(zip(lrs_plot, vals_plot)):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(lrs_plot)))
ax.set_xticklabels([f"{lr:.0e}" for lr in lrs_plot], fontsize=8)
ax.set_xlabel("Learning Rate"); ax.set_ylabel("Val Loss after 6 steps")
ax.set_title(f"FP3: LR Disaster\n({len(crash_lrs)} LR(s) diverged: {crash_lrs})", fontweight="bold")
ax.axhline(base_lr_loss, color="#ea4335", linestyle="--", alpha=0.5, label=f"No-training baseline")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
out_path = "failure_diagnosis.png"
plt.savefig(out_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Chart saved to {out_path}")
free_memory()

Chart saved to failure_diagnosis.png


## Summary

| Failure | Quick Diagnosis | One-line Fix |
|---|---|---|
| **FP1 Forgetting** | Re-eval original task after each new fine-tune | Lower LR for sequential phases |
| **FP2 Overfitting** | Plot train vs val loss — diverging curves | Fewer epochs; more data; dropout |
| **FP3 LR disaster** | Loss NaN or flat after 10 steps | Use 1e-4 for LoRA; add gradient clipping |
| **FP4 Tokenizer** | Batch crashes or padding in outputs | `pad_token = eos_token`; mask labels with -100 |
| **FP5 Contamination** | Val loss suspiciously low | Split before tokenization; hash-deduplicate |
| **FP6 Metric trap** | High ROUGE but bad outputs | Add BERTScore + manual inspection |
| **FP7 MPS gotcha** | RuntimeError on float16 | Use float32; call `mps.empty_cache()` |
| **FP8 Quantization** | Perplexity jumps after quant | Use NF4 + double quant + group_size=128 |
| **FP9 Chat template** | Incoherent or ignored responses | `apply_chat_template()` always |
| **FP10 Grad accum** | Gradients 4× too large | `loss /= GRAD_ACCUM`; scheduler step per optimizer step |

### Next → Step 14: Responsible & Safe Fine-Tuning (`12_responsible_safe_finetuning.ipynb`)

Fine-tuning can silently erode safety guardrails — even on benign data. Next notebook measures and mitigates these risks.